## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

## 2. Load the Datasets

In [ ]:
games_details = pd.read_csv("../data/raw/games_details.csv", low_memory=False)
games = pd.read_csv("../data/raw/games.csv")
players = pd.read_csv("../data/raw/players.csv")
ranking = pd.read_csv("../data/raw/ranking.csv")
teams = pd.read_csv("../data/raw/teams.csv")

## 3. Preview the Datasets

In [ ]:
games_details.head()

In [ ]:
games.head()

In [ ]:
players.head()

In [ ]:
ranking.head()

In [ ]:
teams.head()

## 4. Understand Dataset Shape & Info

In [ ]:
print("Game Details:", games_details.shape)
print("Games:", games.shape)
print("Players:", players.shape)
print("Ranking:", ranking.shape)
print("Teams:", teams.shape)

In [ ]:
games_details.info()

## 5. Clean: games_details

In [ ]:
# Check MIN column format
games_details['MIN'].dropna().head(10)

In [ ]:
# Convert MIN from MM:SS string to float minutes
games_details_cleaned = games_details.copy()

def convert_min_to_float(min_str):
    if isinstance(min_str, str) and ':' in min_str:
        minutes, seconds = min_str.split(':')
        return float(minutes) + float(seconds) / 60
    try:
        return float(min_str)
    except:
        return np.nan

games_details_cleaned['MIN_PLAYED'] = games_details['MIN'].apply(convert_min_to_float)

In [ ]:
# Keep only rows where player actually played
games_details_cleaned = games_details_cleaned[games_details_cleaned['MIN_PLAYED'] > 0]

In [ ]:
# Fill nulls
games_details_cleaned['PLUS_MINUS'] = games_details_cleaned['PLUS_MINUS'].fillna(0)
games_details_cleaned['START_POSITION'] = games_details_cleaned['START_POSITION'].fillna('Bench')
games_details_cleaned['COMMENT'] = games_details_cleaned['COMMENT'].fillna('No Comment')

# Use 'Unknown' instead of 'NA' to prevent pandas re-reading it as NaN
games_details_cleaned['NICKNAME'] = games_details_cleaned['NICKNAME'].fillna('Unknown')

In [ ]:
# Rename columns for clarity
games_details_cleaned.rename(columns={
    'PLAYER_NAME': 'Player',
    'TEAM_ABBREVIATION': 'Team',
    'PTS': 'Points'
}, inplace=True)

In [ ]:
games_details_cleaned.info()

In [ ]:
# Sanity check: no negative values in key stats
(games_details_cleaned[['Points', 'FGA', 'AST', 'REB']] < 0).sum()

In [ ]:
games_details_cleaned['MIN_PLAYED'].describe()

In [ ]:
games_details_cleaned.to_csv("../data/processed/games_details_cleaned.csv", index=False)

## 6. Clean: games

In [ ]:
games.info()

In [ ]:
games_cleaned = games.copy()
games_cleaned['GAME_DATE_EST'] = pd.to_datetime(games_cleaned['GAME_DATE_EST'])

In [ ]:
games_cleaned['GAME_STATUS_TEXT'].value_counts()

In [ ]:
games_cleaned.isnull().sum()

In [ ]:
# Drop 99 rows missing game stats
games_cleaned = games_cleaned.dropna()

In [ ]:
games_cleaned["SEASON"] = games_cleaned["SEASON"].astype(int)

In [ ]:
# Engineer GAME_RESULT column
games_cleaned['GAME_RESULT'] = np.where(
    games_cleaned['HOME_TEAM_WINS'] == 1,
    'HOME_WIN',
    'AWAY_WIN'
)

In [ ]:
# Engineer TOTAL_POINTS column
games_cleaned['TOTAL_POINTS'] = games_cleaned['PTS_home'] + games_cleaned['PTS_away']

In [ ]:
games_cleaned["GAME_STATUS_TEXT"] = games_cleaned["GAME_STATUS_TEXT"].astype("category")

In [ ]:
games_cleaned.info()

In [ ]:
games_cleaned.to_csv("../data/processed/games_cleaned.csv", index=False)

## 7. Clean: players

In [ ]:
players.info()

In [ ]:
# No nulls — export as-is
players.to_csv("../data/processed/players_cleaned.csv", index=False)

## 8. Clean: ranking

In [ ]:
ranking.info()

In [ ]:
ranking_cleaned = ranking.copy()
ranking_cleaned["RETURNTOPLAY"] = ranking_cleaned["RETURNTOPLAY"].fillna(0)

In [ ]:
ranking_cleaned["STANDINGSDATE"] = pd.to_datetime(ranking_cleaned["STANDINGSDATE"])

In [ ]:
# Split home and road records into numeric columns
ranking_cleaned[["HOME_WINS", "HOME_LOSSES"]] = ranking_cleaned["HOME_RECORD"].str.split("-", expand=True).astype(int)
ranking_cleaned[["ROAD_WINS", "ROAD_LOSSES"]] = ranking_cleaned["ROAD_RECORD"].str.split("-", expand=True).astype(int)

In [ ]:
# Extract season year and phase from SEASON_ID (e.g. 22022 -> year=2022, phase=2)
ranking_cleaned["SEASON_YEAR"] = ranking_cleaned["SEASON_ID"] % 10000
ranking_cleaned["SEASON_PHASE"] = ranking_cleaned["SEASON_ID"] // 10000

In [ ]:
ranking_cleaned.info()

In [ ]:
ranking_cleaned.to_csv("../data/processed/ranking_cleaned.csv", index=False)

## 9. Clean: teams & Merge with Games

In [ ]:
teams.info()

In [ ]:
# Merge home team info onto games
games_with_teams = games_cleaned.merge(
    teams,
    left_on="HOME_TEAM_ID",
    right_on="TEAM_ID",
    how="left"
).rename(columns={
    "CITY": "HOME_CITY",
    "NICKNAME": "HOME_TEAM_NAME"
}).drop(columns=["TEAM_ID"])

In [ ]:
# Merge away team info onto games
games_with_teams = games_with_teams.merge(
    teams,
    left_on="VISITOR_TEAM_ID",
    right_on="TEAM_ID",
    how="left"
).rename(columns={
    "CITY": "AWAY_CITY",
    "NICKNAME": "AWAY_TEAM_NAME"
}).drop(columns=["TEAM_ID"])

In [ ]:
teams.to_csv("../data/processed/teams_cleaned.csv", index=False)

In [ ]:
games_with_teams.to_csv("../data/processed/games_with_teams.csv", index=False)
print("All cleaned files exported to data/processed/")